In [1]:
import os
import numpy as np
import cv2
from sklearn.model_selection import train_test_split
from numpy.linalg import inv

In [2]:
labels = ["Non Demented", "Very mild Dementia", "Mild Dementia", "Moderate Dementia"]
data_dir = "oasis"
img_size = (224, 224)
num_classes = len(labels)

X, y = [], []

# Load Dataset
for label in labels:
    class_path = os.path.join(data_dir, label)

    for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path, img_name)

        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        # Resize
        img_resize = cv2.resize(img, img_size)

        # Normalisasi
        img_norm = img_resize / 255.0

        X.append(img_norm)
        y.append(labels.index(label))

X = np.expand_dims(X, axis=-1)
X = np.array(X)
y = np.array(y, dtype=int)

y_onehot = np.zeros((len(y), num_classes), dtype=int)

for i, label in enumerate(y):
    y_onehot[i, label] = 1

In [3]:
def relu(x):
    return np.maximum(0, x)

def conv2d_single(input_img, kernels, bias, activation="relu"):
    H, W, C = input_img.shape
    F, kH, kW, _ = kernels.shape

    out_H = H - kH + 1
    out_W = W - kW + 1

    output = np.zeros((out_H, out_W, F))

    for f in range(F):
        for i in range(out_H):
            for j in range(out_W):
                region = input_img[i:i+kH, j:j+kW, :]
                output[i, j, f] = np.sum(region * kernels[f]) + bias[f]

    if activation == "relu":
        output = relu(output)

    return output

def conv2d_batch(X, filters, kernel_size=(3,3)):
    N, H, W, C = X.shape
    kH, kW = kernel_size

    # inisialisasi bobot
    kernels = np.random.randn(filters, kH, kW, C) * 0.1    
    bias = np.random.randn(filters) * 0.1

    outputs = []

    for n in range(N):
        out = conv2d_single(X[n], kernels, bias)
        outputs.append(out)

    return np.array(outputs), kernels, bias

In [4]:
def maxpool2d_single(input_img, pool_size=(2,2), stride=2):
    H, W, C = input_img.shape
    pH, pW = pool_size

    out_H = (H - pH) // stride + 1
    out_W = (W - pW) // stride + 1

    output = np.zeros((out_H, out_W, C))

    for c in range(C):
        for i in range(out_H):
            for j in range(out_W):
                h_start = i * stride
                w_start = j * stride

                region = input_img[
                    h_start:h_start+pH,
                    w_start:w_start+pW,
                    c
                ]

                output[i, j, c] = np.max(region)

    return output

def maxpool2d_batch(X, pool_size=(2,2), stride=2):
    N = X.shape[0]
    outputs = []

    for n in range(N):
        out = maxpool2d_single(X[n], pool_size, stride)
        outputs.append(out)

    return np.array(outputs)

In [5]:
def global_avg_pool_single(input_img):
    # input: (H, W, C)
    return np.mean(input_img, axis=(0,1))

def global_avg_pool_batch(X):
    N = X.shape[0]
    C = X.shape[3]

    output = np.zeros((N, C))

    for n in range(N):
        output[n] = np.mean(X[n], axis=(0,1))

    return output

In [6]:
class ELM:
    def __init__(self, input_dim, hidden_dim, output_dim, activation="tanh", reg=1e-3):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        self.reg = reg
        self.W = np.random.randn(self.input_dim, self.hidden_dim)
        self.b = np.random.randn(self.hidden_dim)

        if activation == "sigmoid":
            self.activation = lambda x: 1.0 / (1.0 + np.exp(-x))
        elif activation == "relu":
            self.activation = lambda x: np.maximum(0, x)
        elif activation == "tanh":
            self.activation = lambda x: np.tanh(x)
        else:
            raise ValueError("Unsupported activation function")

    def _softmax(self, x):
        exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
        return exp_x / np.sum(exp_x, axis=1, keepdims=True)

    def fit(self, X, y_int):
        H = self.activation(np.dot(X, self.W) + self.b)
        HtH = H.T @ H
        I = np.eye(HtH.shape[0])
        self.beta = inv(HtH + self.reg * I) @ H.T @ y_int

    def predict(self, X):
        H = self.activation(np.dot(X, self.W) + self.b)
        y_raw = np.dot(H, self.beta)
        y_prob = self._softmax(y_raw)
        return np.argmax(y_prob, axis=1), y_prob

In [7]:
def categorical_cross_entropy(y_true, y_prob, eps=1e-12):
    y_prob = np.clip(y_prob, eps, 1. - eps)
    loss = -np.sum(y_true * np.log(y_prob), axis=1)
    return np.mean(loss)

In [8]:
def validasi(y_true, y_pred, num_classes):
    precision_list = []
    recall_list = []
    f1_list = []

    accuracy = np.sum(y_true == y_pred) / len(y_true)
    for c in range(num_classes):
        TP = np.sum((y_true == c) & (y_pred == c))
        FP = np.sum((y_true != c) & (y_pred == c))
        FN = np.sum((y_true == c) & (y_pred != c))

        precision = TP / (TP + FP + 1e-10)
        recall = TP / (TP + FN + 1e-10)
        f1 = 2 * precision * recall / (precision + recall + 1e-10)

        precision_list.append(precision)
        recall_list.append(recall)
        f1_list.append(f1)
    precision = np.mean(precision_list)
    recall = np.mean(recall_list)
    f1_score = np.mean(f1_list)

    return accuracy, precision, recall, f1_score

In [9]:
def CNN_ELM(jumlah_kernel, X, y, epsilon, max_epoch, aktivasi):
    # ================= Feature Extraction =================
    out, conv_kernel, conv_bias = conv2d_batch(X, filters=jumlah_kernel)
    pooled = maxpool2d_batch(out, pool_size=(2,2), stride=2)
    gap = global_avg_pool_batch(pooled)

    # ================= Split Data =================
    X_train, X_test, y_train, y_test = train_test_split(gap, y, test_size=0.2, random_state=42, stratify=np.argmax(y_onehot, axis=1))

    # ================= Training =================
    error = np.inf
    epoch = 1
    num_classes = 4
    while epoch <= max_epoch and error > epsilon:
        elm = ELM(
            input_dim=X_train.shape[1],
            hidden_dim=4000,
            output_dim=num_classes,
            activation=aktivasi
        )

        elm.fit(X_train, y_train)

        _, y_prob = elm.predict(X_train)

        error = categorical_cross_entropy(
            y_true=y_train,
            y_prob=y_prob)

        epoch += 1

    # ================= Evaluation =================
    y_val_train, _ = elm.predict(X_train)
    y_true_train = np.argmax(y_train, axis=1)
    akurasi_train, presisi_train, recall_train, f1score_train = validasi(
    y_pred=y_val_train,
    y_true=y_true_train,
    num_classes=num_classes)
    
    y_val, _ = elm.predict(X_test)

    y_true = np.argmax(y_test, axis=1)

    akurasi, presisi, recall, f1score = validasi(
    y_pred=y_val,
    y_true=y_true,
    num_classes=num_classes)

    print("=== Train ===")
    print(f"Kernel: {jumlah_kernel}- Epoch: {epoch} - Loss: {error:.6f}")
    print(f"Akurasi : {akurasi_train:.6f}")
    print(f"Presisi : {presisi_train:.6f}")
    print(f"Recall  : {recall_train:.6f}")
    print(f"F1-score: {f1score_train:.6f}")

    print("=== Test ===")
    print(f"Akurasi : {akurasi:.6f}")
    print(f"Presisi : {presisi:.6f}")
    print(f"Recall  : {recall:.6f}")
    print(f"F1-score: {f1score:.6f}")

    return conv_kernel, conv_bias, gap, elm.W, elm.beta, elm.b

#sigmoid

In [ ]:
# ================= Kernel 8 =================

conv_kernel8s, conv_bias8s, gap8s, elm_W8s, elm_beta8s, elm_b8s = CNN_ELM(jumlah_kernel=8, X=X, y=y_onehot, epsilon=0.9, max_epoch=30, aktivasi="sigmoid")

=== Train ===
Kernel: 8- Epoch: 31 - Loss: 1.167550
Akurasi : 0.743750
Presisi : 0.740219
Recall  : 0.743750
F1-score: 0.734687
=== Test ===
Akurasi : 0.812500
Presisi : 0.816446
Recall  : 0.812500
F1-score: 0.809372


In [11]:
# ================= Kernel 16 =================

conv_kernel16s, conv_bias16s, gap16s, elm_W16s, elm_beta16s, elm_b16s = CNN_ELM(jumlah_kernel=16, X=X, y=y_onehot, epsilon=0.9, max_epoch=30, aktivasi="sigmoid")

=== Train ===
Kernel: 16- Epoch: 31 - Loss: 1.118668
Akurasi : 0.862500
Presisi : 0.870705
Recall  : 0.862500
F1-score: 0.862148
=== Test ===
Akurasi : 0.837500
Presisi : 0.850000
Recall  : 0.837500
F1-score: 0.840079


In [10]:
# ================= Kernel 32 =================

conv_kernel32s, conv_bias32s, gap32s, elm_W32s, elm_beta32s, elm_b32s = CNN_ELM(jumlah_kernel=32, X=X, y=y_onehot, epsilon=0.09, max_epoch=30, aktivasi="sigmoid")

=== Train ===
Kernel: 32- Epoch: 31 - Loss: 1.065634
Akurasi : 0.887500
Presisi : 0.893648
Recall  : 0.887500
F1-score: 0.887156
=== Test ===
Akurasi : 0.900000
Presisi : 0.910949
Recall  : 0.900000
F1-score: 0.900608


#tanh

In [ ]:
# ================= Kernel 8 =================

conv_kernel8t, conv_bias8t, gap8t, elm_W8t, elm_beta8t, elm_b8t = CNN_ELM(jumlah_kernel=8, X=X, y=y_onehot, epsilon=0.9, max_epoch=30, aktivasi="tanh")

=== Train ===
Kernel: 8- Epoch: 31 - Loss: 1.100774
Akurasi : 0.756250
Presisi : 0.749813
Recall  : 0.756250
F1-score: 0.751129
=== Test ===
Akurasi : 0.775000
Presisi : 0.770439
Recall  : 0.775000
F1-score: 0.771517


In [13]:
# ================= Kernel 16 =================

conv_kernel16t, conv_bias16t, gap16t, elm_W16t, elm_beta16t, elm_b16t = CNN_ELM(jumlah_kernel=16, X=X, y=y_onehot, epsilon=0.9, max_epoch=30, aktivasi="tanh")

=== Train ===
Kernel: 16- Epoch: 31 - Loss: 1.003097
Akurasi : 0.890625
Presisi : 0.889834
Recall  : 0.890625
F1-score: 0.889210
=== Test ===
Akurasi : 0.850000
Presisi : 0.852020
Recall  : 0.850000
F1-score: 0.849875


In [15]:
# ================= Kernel 32 =================

conv_kernel32t, conv_bias32t, gap32t, elm_W32t, elm_beta32t, elm_b32t = CNN_ELM(jumlah_kernel=32, X=X, y=y_onehot, epsilon=0.09, max_epoch=30, aktivasi="tanh")

=== Train ===
Kernel: 32- Epoch: 31 - Loss: 0.888902
Akurasi : 0.987500
Presisi : 0.987654
Recall  : 0.987500
F1-score: 0.987539
=== Test ===
Akurasi : 0.962500
Presisi : 0.964286
Recall  : 0.962500
F1-score: 0.962789
